# Lab 3 — Logistic Regression, Softmax and Naive Bayes from Scratch

**Course:** Machine Learning · BSc in Artificial Intelligence · Central Asian University
**Lecture:** Lecture 3 — Classification with Probabilities
**Lecturer:** Behnam Kiani

| | |
|---|---|
| **Student name** | *your name* |
| **Group** | *your group* |
| **Date** | *dd.mm.yyyy* |

---

### What you will learn in this lab

By the end of this notebook you will be able to:

1. Implement the **sigmoid** (in a numerically stable way) and the **log loss**, and check the **gradient** of logistic regression numerically.
2. Train **logistic regression with gradient descent** from scratch and match scikit-learn.
3. Use predicted **probabilities**: build a **confusion matrix**, compute **precision and recall**, choose a **threshold**, and read coefficients as **odds ratios**.
4. Implement **softmax regression** for three classes with the **cross-entropy** loss.
5. Implement **Gaussian** and **multinomial Naive Bayes** from scratch, including **Laplace smoothing**, and build a working **SMS spam filter**.

## How to use this notebook — read this first

**Cells.** This notebook has two kinds of cells:
- **Text (Markdown) cells** like this one explain what to do. Double-click to edit; press **Shift + Enter** to display.
- **Code cells** contain Python. Run a cell with **Shift + Enter**. Run cells **in order, from top to bottom** — later cells use variables created by earlier ones.

**How each task is organised.** Every code cell has a text cell above it with the same structure:

| Label | Meaning |
|---|---|
| **What this cell does** | Explains the code in the next cell. |
| **Your task** | What *you* must do. "Just run it" means there is nothing to write. |
| `# TODO` | A line in the code that you must complete. Replace `None`, `...` or `pass`. |
| **Hint** | A tip if you are stuck. Try without it first. |
| **Expected result** | What you should see, so you can check yourself. Small differences in the last digit are fine. |
| `# CHECK` cell | Tests your work automatically. It prints **✓** when your answer is correct and stops with an `AssertionError` that explains the problem when it is not. **Do not edit CHECK cells.** |
| **Your answer** | A text cell where you explain something in your own words (2–5 sentences). |

**Rules for this lab**
- Parts A, B, D and the "from scratch" tasks of Part E: use **NumPy only**. scikit-learn is used **only to check** your results.
- Part C and the comparison tasks of Part E: scikit-learn is allowed.
- If you get an error, read the **last line** of the error message first, then print the `.shape` of your arrays.

**Before you submit:** **Runtime → Restart and run all** (Colab) or **Kernel → Restart & Run All** (Jupyter). The notebook must run from top to bottom with no errors and every CHECK cell must print ✓. Rename the file to `Lab03_<Surname>_<Name>.ipynb`.

**Academic integrity.** You may discuss ideas with classmates, but every line of code and every written answer must be your own and you must be able to explain it.

## Contents

- **Part A** — The sigmoid, the log loss and the gradient (≈ 30 min)
- **Part B** — Logistic regression from scratch on the penguins (≈ 45 min)
- **Part C** — Breast cancer: probabilities, thresholds and odds ratios (≈ 35 min)
- **Part D** — Softmax regression for three species (≈ 40 min)
- **Part E** — Naive Bayes: Gaussian and a spam filter from scratch (≈ 50 min)
- **Part F** — Reflection
- **Submission checklist**

### Key formulas from the lecture

| Quantity | Formula |
|---|---|
| Sigmoid | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ |
| Logistic model | $p = P(y=1\mid\mathbf{x}) = \sigma(\mathbf{w}^\top\mathbf{x})$ |
| Log loss (binary cross-entropy) | $J(\mathbf{w}) = -\dfrac{1}{n}\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big]$ |
| Gradient of the log loss | $\nabla J(\mathbf{w}) = \dfrac{1}{n}X^\top\big(\sigma(X\mathbf{w})-\mathbf{y}\big)$ |
| L2-regularised gradient | $\nabla J(\mathbf{w}) + \lambda\,\mathbf{w}$ (no penalty on the bias); scikit-learn's `C` corresponds to $\lambda = 1/(C\,n)$ |
| Odds ratio | $e^{w_j}$ = factor by which the odds change when $x_j$ increases by 1 |
| Softmax | $P(y=k\mid\mathbf{x}) = \dfrac{e^{z_k}}{\sum_j e^{z_j}}$, $z_k = \mathbf{w}_k^\top\mathbf{x}$ |
| Cross-entropy | $J(W) = -\dfrac{1}{n}\sum_i\sum_k Y_{ik}\log P_{ik}$, gradient $\dfrac{1}{n}X^\top(P-Y)$ |
| Naive Bayes | $\hat y = \arg\max_k \big[\log P(y=k) + \sum_j \log P(x_j\mid y=k)\big]$ |
| Laplace smoothing | $\hat P(w\mid c) = \dfrac{\text{count}(w,c)+\alpha}{\sum_v \text{count}(v,c)+\alpha\lvert V\rvert}$ |

## Setup

**What this cell does:** imports the libraries used in the lab and sets NumPy to print 4 decimals.
- `numpy` (`np`) — arrays and linear algebra; `pandas` (`pd`) — loading data; `matplotlib.pyplot` (`plt`) — plots
- from scikit-learn: tools to split data, scale features, and reference models we use **only to check** our own code.

**Your task:** just run it. If an import fails, tell your lab assistant.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import log_loss, confusion_matrix
from sklearn.datasets import load_breast_cancer

np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__, "| pandas", pd.__version__)

---
# Part A — The sigmoid, the log loss and the gradient

Logistic regression has three ingredients: the **sigmoid**, which turns a score into a probability; the **log loss**, which measures how good the probabilities are; and the **gradient** of the loss, which gradient descent needs. You will write all three and test them.

### A1. The sigmoid function

**What you need to know:** $\sigma(z) = \dfrac{1}{1+e^{-z}}$. It maps any real number to $(0, 1)$, with $\sigma(0) = 0.5$.

**What this cell must do:** define a function `sigmoid(z)` that works on a single number **and** on a NumPy array (element by element), then print $\sigma$ at $-2, 0, 2$.

**Your task:** replace `None` with the formula.

**Hint:** use `np.exp`, not `math.exp` — `np.exp` works on whole arrays.

**Expected result:** `[0.1192 0.5    0.8808]`

In [ ]:
def sigmoid(z):
    return None   # TODO: 1 / (1 + e^(-z))

print(sigmoid(np.array([-2.0, 0.0, 2.0])))

In [ ]:
# CHECK A1 — do not edit
assert sigmoid(np.array([0.0])) is not None, "Complete sigmoid first."
assert np.allclose(sigmoid(np.array([-2.0, 0.0, 2.0])), [0.11920292, 0.5, 0.88079708])
print("✓ A1")

### A2. A numerically stable sigmoid

**What this cell does:** the first line computes $\sigma(-1000)$ with your function. Because $e^{1000}$ is larger than the biggest number a computer can store, NumPy prints an **overflow warning**. Large scores really happen during training (Part B5 shows weights growing very large).

**Your task:** write `stable_sigmoid(z)` that never overflows. The cell then **replaces** `sigmoid` by your stable version, so the rest of the lab uses it.

**Hint:** clip the scores first: `z = np.clip(z, -500, 500)`. $\sigma(\pm 500)$ is already 0 or 1 to machine precision, so clipping changes nothing else.

**Expected result:** a warning from the first line; then `[0.  0.5 1. ]` with **no** warning.

In [ ]:
if sigmoid(np.array([0.0])) is not None:
    print("old sigmoid(-1000):", sigmoid(np.array([-1000.0])))

def stable_sigmoid(z):
    z = z            # TODO: clip z to the range [-500, 500]
    return None      # TODO: the sigmoid formula

if stable_sigmoid(np.array([0.0])) is not None:
    sigmoid = stable_sigmoid
    print(sigmoid(np.array([-1000.0, 0.0, 1000.0])))

In [ ]:
# CHECK A2 — do not edit
assert stable_sigmoid(np.array([0.0])) is not None, "Complete stable_sigmoid first."
with np.errstate(over="raise"):
    out = stable_sigmoid(np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0]))
assert np.allclose(out, [0, 0.11920292, 0.5, 0.88079708, 1]) and not np.isnan(out).any()
print("✓ A2 — no overflow")

### A3. The log loss

**What you need to know:** for labels $y_i \in \{0,1\}$ and predicted probabilities $p_i = P(y_i = 1)$:
$$J = -\frac{1}{n}\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big].$$
For each example this is $-\log(\text{probability given to the true class})$.

**What this cell must do:** define `log_loss_np(y, p)` and evaluate it on five small examples.

**Your task:** complete the formula. To avoid $\log 0$, the probabilities are first clipped to $[\varepsilon, 1-\varepsilon]$ (already written).

**Hint:** `np.mean(...)` of the bracket, with a minus sign in front. `*` multiplies element by element.

**Expected result:** `0.272`

In [ ]:
def log_loss_np(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return None   # TODO: the log loss formula

y_small = np.array([1, 0, 1, 1, 0])
p_small = np.array([0.9, 0.2, 0.6, 0.99, 0.4])
if log_loss_np(y_small, p_small) is not None:
    print(round(log_loss_np(y_small, p_small), 4))

In [ ]:
# CHECK A3 — do not edit
assert log_loss_np(y_small, p_small) is not None, "Complete log_loss_np first."
assert np.isclose(log_loss_np(y_small, p_small), log_loss(y_small, p_small))
assert np.isclose(log_loss_np(np.array([1, 0]), np.array([0.5, 0.5])), np.log(2))
print("✓ A3 — matches scikit-learn's log_loss")

**Question A3:** in the five examples, which single example contributes most to the loss, and why? What is the log loss of a model that always predicts $p = 0.5$?

**Your answer (A3):**

*Write your answer here — double-click this cell to edit it.*

### A4. The gradient, checked numerically

**What you need to know:** the gradient of the log loss is
$$\nabla J(\mathbf{w}) = \frac{1}{n}A^\top\big(\sigma(A\mathbf{w}) - \mathbf{y}\big),$$
where $A$ is the design matrix (a first column of ones, then the features). A **numerical gradient** approximates each partial derivative with a tiny step $h$:
$\dfrac{\partial J}{\partial w_j} \approx \dfrac{J(\mathbf{w} + h\mathbf{e}_j) - J(\mathbf{w} - h\mathbf{e}_j)}{2h}$. If both agree, your formula is right.

**What this cell must do:** create a small random data set, define `loss_logistic(w, A, y)` (already written) and `gradient(w, A, y)` (your task), then compare with the numerical gradient.

**Your task:** complete `gradient`.

**Hint:** `A.T @ (sigmoid(A @ w) - y) / len(y)`. Check shapes: `A` is (50, 3), `w` is (3,), the result must be (3,).

**Expected result:** two almost identical vectors and a difference smaller than `1e-8`.

In [ ]:
rng = np.random.default_rng(0)
A_toy = np.c_[np.ones(50), rng.normal(size=(50, 2))]   # 50 examples: bias column + 2 features
y_toy = rng.integers(0, 2, size=50)
w_toy = np.array([0.1, -0.2, 0.3])

def loss_logistic(w, A, y):
    return log_loss_np(y, sigmoid(A @ w))

def gradient(w, A, y):
    return None   # TODO: (1/n) A^T (sigmoid(A w) - y)

def numerical_gradient(f, w, h=1e-5):
    g = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = h
        g[j] = (f(w + e) - f(w - e)) / (2 * h)
    return g

g_num = None
if sigmoid(np.array([0.0])) is not None and log_loss_np(y_small, p_small) is not None:
    g_num = numerical_gradient(lambda w: loss_logistic(w, A_toy, y_toy), w_toy)
    print("numerical:", g_num)
if g_num is not None and gradient(w_toy, A_toy, y_toy) is not None:
    g_ana = gradient(w_toy, A_toy, y_toy)
    print("analytic :", g_ana)
    print("max difference:", np.abs(g_num - g_ana).max())

In [ ]:
# CHECK A4 — do not edit
assert g_num is not None, "Complete A1–A3 first."
assert gradient(w_toy, A_toy, y_toy) is not None, "Complete gradient first."
assert gradient(w_toy, A_toy, y_toy).shape == (3,), "The gradient must have one entry per weight."
assert np.allclose(gradient(w_toy, A_toy, y_toy), g_num, atol=1e-7), "Analytic and numerical gradients differ."
print("✓ A4 — your gradient is correct")

**Question A4:** in the lecture, the chain rule gave $\partial\ell_i/\partial\mathbf{w} = (p_i - y_i)\,\mathbf{x}_i$. In your own words: what happens to the gradient contribution of an example that the model already predicts perfectly? And of one it gets badly wrong?

**Your answer (A4):**

*Write your answer here — double-click this cell to edit it.*

---
# Part B — Logistic regression from scratch on the penguins

**Task:** is a penguin an **Adelie** ($y = 0$) or a **Chinstrap** ($y = 1$), given its bill length and bill depth? This is the running example of the lecture.

### B0. Load and prepare the data

**What this cell does:**
1. Downloads `penguins.csv` and keeps the Adelie and Chinstrap penguins with both bill measurements.
2. Builds `X` (two columns: bill length, bill depth, in mm) and `y` (1 = Chinstrap).
3. Splits into 75 % training and 25 % test data, keeping the class proportions (`stratify=y`).
4. **Standardises** the features with the **training** mean and standard deviation (no leakage), and adds a column of ones for the bias: `A_train`, `A_test`.

**Your task:** just run it and read the printed shapes.

**Expected result:** `A_train (164, 3) | A_test (55, 3) | fraction Chinstrap in training: 0.311`

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(url).dropna(subset=["bill_length_mm", "bill_depth_mm"])

two = penguins[penguins["species"].isin(["Adelie", "Chinstrap"])]
X = two[["bill_length_mm", "bill_depth_mm"]].to_numpy()
y = (two["species"] == "Chinstrap").astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
A_train = np.c_[np.ones(len(X_train)), (X_train - mu) / sd]
A_test = np.c_[np.ones(len(X_test)), (X_test - mu) / sd]
print("A_train", A_train.shape, "| A_test", A_test.shape, "| fraction Chinstrap in training:", round(y_train.mean(), 3))

### B1. Gradient descent for logistic regression

**What you need to know:** the update is $\mathbf{w} \leftarrow \mathbf{w} - \eta\,\big(\nabla J(\mathbf{w}) + \lambda\,\mathbf{w}_{\text{no bias}}\big)$. The small L2 penalty $\lambda$ makes our model identical to scikit-learn's default `LogisticRegression(C=1.0)`, which uses $\lambda = 1/(C\,n)$. The bias $w_0$ is **not** penalised.

**What this cell must do:** define `fit_logistic(A, y, eta, steps, lam)`, which starts from $\mathbf{w} = \mathbf{0}$, runs `steps` updates and returns the weights and the list of training losses. Then train on the penguins with $\eta = 0.5$, 2000 steps and $\lambda = 1/n$.

**Your task:** complete the two `TODO` lines inside the loop.

**Hint:**
- `penalty = lam * np.r_[0, w[1:]]` is already written — it is $\lambda\mathbf{w}$ with a 0 for the bias.
- `g = gradient(w, A, y) + penalty`, then `w = w - eta * g`.

**Expected result:** `w = [-1.8483  3.8418 -1.1767]`, final training loss `0.0865`. Positive weight for length (longer bills → Chinstrap), negative for depth.

In [ ]:
def fit_logistic(A, y, eta=0.5, steps=2000, lam=0.0):
    w = np.zeros(A.shape[1])
    losses = []
    for step in range(steps):
        penalty = lam * np.r_[0, w[1:]]
        g = None          # TODO: gradient of the loss plus the penalty
        w = w             # TODO: gradient-descent update
        losses.append(loss_logistic(w, A, y))
    return w, losses

n_train = len(y_train)
w_gd = None
if g_num is not None and gradient(w_toy, A_toy, y_toy) is not None:
    w_gd, losses_gd = fit_logistic(A_train, y_train, eta=0.5, steps=2000, lam=1 / n_train)
    print("w =", w_gd, "| final training loss:", round(losses_gd[-1], 4))

**What this cell does:** fits scikit-learn's `LogisticRegression(C=1.0)` on the same standardised features and checks that your weights agree with it to two decimals.

In [ ]:
# CHECK B1 — do not edit
assert w_gd is not None, "Complete Part A and fit_logistic first."
sk = LogisticRegression(C=1.0).fit((X_train - mu) / sd, y_train)
w_sk = np.r_[sk.intercept_, sk.coef_[0]]
print("scikit-learn:", w_sk)
assert np.allclose(w_gd, w_sk, atol=1e-2), "Your weights differ from scikit-learn — check the update and the penalty."
print("✓ B1 — gradient descent matches scikit-learn")

### B2. Predictions and accuracy

**What this cell must do:** define `predict(w, A, t=0.5)` that returns 1 when the predicted probability is at least the threshold `t`, else 0; then print the training and test accuracy.

**Your task:** complete `predict` and the two accuracies.

**Hint:** `(sigmoid(A @ w) >= t).astype(int)`; accuracy is `np.mean(prediction == truth)`.

**Expected result:** training accuracy `0.970`, test accuracy `0.964` (53 of 55 test penguins).

In [ ]:
def predict(w, A, t=0.5):
    return None   # TODO

acc_train = None  # TODO
acc_test = None   # TODO
if w_gd is not None and acc_test is not None:
    print(f"training accuracy {acc_train:.3f} | test accuracy {acc_test:.3f}")

### B3. The loss curve and the decision boundary

**What you need to know:** in standardised units $s_1, s_2$, the boundary is where the score is zero: $w_0 + w_1 s_1 + w_2 s_2 = 0$, so $s_2 = -(w_0 + w_1 s_1)/w_2$. The code converts back to millimetres with $x = \mu + \sigma s$.

**What this cell must do:** plot (left) the training loss against the step, and (right) the test penguins with the decision boundary.

**Your task:** complete the line `s2 = None` using the formula above.

**Expected result:** the loss starts at 0.693 ($\log 2$) and decreases quickly; the black line separates blue Adelies from orange Chinstraps with only 2 test penguins on the wrong side.

In [ ]:
if acc_test is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(losses_gd, color="#6250d6")
    axs[0].set_xlabel("step"); axs[0].set_ylabel("training log loss"); axs[0].set_title("Gradient descent")

    length = np.linspace(32, 58, 100)
    s1 = (length - mu[0]) / sd[0]
    s2 = None   # TODO: standardised depth on the boundary
    if s2 is not None:
        depth = mu[1] + sd[1] * s2
        axs[1].plot(length, depth, color="black", lw=2, label="boundary P = 0.5")
    axs[1].scatter(*X_test[y_test == 0].T, color="#2a78d6", label="Adelie")
    axs[1].scatter(*X_test[y_test == 1].T, color="#eb6834", label="Chinstrap")
    axs[1].set_ylim(15, 22); axs[1].set_xlabel("bill length (mm)"); axs[1].set_ylabel("bill depth (mm)")
    axs[1].legend(); axs[1].set_title("Test penguins and decision boundary")
    plt.show()

### B4. Three learning rates

**What this cell does:** trains for 300 steps with $\eta = 0.05$, $0.5$ and $5$ and prints the final training loss of each.

**Your task:** just run it, then answer the question below.

**Expected result:** `0.1782`, `0.0893`, `0.0865` — all three converge; the largest $\eta$ is fastest here.

In [ ]:
if acc_test is not None:
    for eta in [0.05, 0.5, 5.0]:
        _, ls = fit_logistic(A_train, y_train, eta=eta, steps=300, lam=1 / n_train)
        print(f"eta = {eta:<5} loss after 300 steps = {ls[-1]:.4f}")

**Question B4:** last week, a learning rate of 1.05 made linear regression explode. Here even $\eta = 5$ works. Using the shape of the sigmoid, explain why logistic regression is more tolerant of large steps (with standardised features). Would it be safe to use $\eta = 500$?

**Your answer (B4):**

*Write your answer here — double-click this cell to edit it.*

### B5. When the classes are perfectly separable

**What this cell does:** uses **Adelie vs Gentoo**, which can be separated perfectly by a straight line. It fits scikit-learn's logistic regression with weaker and weaker regularisation (larger `C`) and prints the largest weight.

**Your task:** just run it, then answer the question.

**Expected result:** largest |w| ≈ `2.9`, `8.7`, `24.4`, `43.3` — and training accuracy 100 % from `C = 100` on.

In [ ]:
ag = penguins[penguins["species"].isin(["Adelie", "Gentoo"])]
X_ag = StandardScaler().fit_transform(ag[["bill_length_mm", "bill_depth_mm"]])
y_ag = (ag["species"] == "Gentoo").astype(int)
for C in [1, 100, 1e4, 1e6]:
    m = LogisticRegression(C=C, max_iter=100000, tol=1e-10).fit(X_ag, y_ag)
    w_all = np.r_[m.intercept_, m.coef_[0]]
    print(f"C = {C:>9.0f} | largest |w| = {np.abs(w_all).max():6.1f} | training accuracy = {m.score(X_ag, y_ag):.3f}")

**Question B5:** why do the weights keep growing when the data are separable? What is wrong with a model whose weights are this large, even if its accuracy is 100 %? Which scikit-learn setting prevents it?

**Your answer (B5):**

*Write your answer here — double-click this cell to edit it.*

---
# Part C — Breast cancer: probabilities, thresholds and odds ratios

**Data:** 569 biopsies with 30 measurements of the cell nuclei (Street, Wolberg & Mangasarian, 1993), built into scikit-learn. We code **malignant = 1**. In this part you may use scikit-learn for the model; you write the **evaluation** yourself.

### C0. Load and split

**What this cell does:** loads the data, recodes the target so that 1 = malignant, and splits 75 % / 25 % with `random_state=42`, keeping class proportions.

**Your task:** just run it.

**Expected result:** `training (426, 30) | test (143, 30) | malignant in test: 53`

In [ ]:
bc = load_breast_cancer()
X_bc = bc.data
y_bc = 1 - bc.target          # scikit-learn codes benign = 1; we want malignant = 1
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc)
print("training", Xb_train.shape, "| test", Xb_test.shape, "| malignant in test:", yb_test.sum())

### C1. A logistic-regression pipeline

**What this cell must do:** build a **pipeline** that standardises the features and then fits logistic regression, train it on the training set, compute the predicted probabilities of malignancy on the test set (`p_bc`) and the test accuracy at threshold 0.5.

**Your task:** complete the three `TODO` lines.

**Hint:**
- `make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))`
- `clf.predict_proba(Xb_test)[:, 1]` gives P(malignant) — column 1 is class 1.

**Expected result:** test accuracy `0.965`.

In [ ]:
clf = None            # TODO: the pipeline
p_bc = None           # TODO: probabilities of class 1 on the test set (after fitting)
acc_bc = None         # TODO: accuracy at threshold 0.5
if acc_bc is not None:
    print("test accuracy:", round(acc_bc, 3))

In [ ]:
# CHECK C1 — do not edit
assert p_bc is not None and acc_bc is not None, "Complete C1 first."
assert p_bc.shape == (143,) and np.all((p_bc >= 0) & (p_bc <= 1)), "p_bc must hold 143 probabilities."
assert np.isclose(acc_bc, 0.965, atol=0.01)
print("✓ C1")

### C2. The confusion matrix from scratch

**What you need to know:** rows = truth, columns = prediction:

| | predicted 0 | predicted 1 |
|---|---|---|
| **true 0** | TN | FP |
| **true 1** | FN | TP |

**What this cell must do:** define `confusion_2x2(y_true, y_pred)` that returns the 2 × 2 NumPy array above, then apply it at threshold 0.5.

**Your task:** compute the four counts.

**Hint:** `TN = np.sum((y_true == 0) & (y_pred == 0))` — the others are similar.

**Expected result:** `[[89  1] [ 4 49]]` — 4 missed cancers, 1 false alarm.

In [ ]:
def confusion_2x2(y_true, y_pred):
    TN = None   # TODO
    FP = None   # TODO
    FN = None   # TODO
    TP = None   # TODO
    if TN is None:
        return None
    return np.array([[TN, FP], [FN, TP]])

if p_bc is not None and confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)) is not None:
    print(confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)))

In [ ]:
# CHECK C2 — do not edit
assert p_bc is not None and confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)) is not None, "Complete C1 and C2 first."
assert np.array_equal(confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)), confusion_matrix(yb_test, p_bc >= 0.5))
print("✓ C2 — matches scikit-learn's confusion_matrix")

### C3. Precision and recall

**What you need to know:**
- **Recall** (sensitivity) $= \dfrac{TP}{TP + FN}$: of the real cancers, the fraction we found.
- **Precision** $= \dfrac{TP}{TP + FP}$: of the cases we called cancer, the fraction that really were.

**What this cell must do:** define `recall_precision(cm)` that takes a 2 × 2 confusion matrix and returns `(recall, precision)`.

**Your task:** complete the two formulas.

**Hint:** with `cm = [[TN, FP], [FN, TP]]`, `TP = cm[1, 1]`, `FN = cm[1, 0]`, `FP = cm[0, 1]`.

**Expected result:** `recall 0.925 | precision 0.980`

In [ ]:
def recall_precision(cm):
    recall = None      # TODO
    precision = None   # TODO
    return recall, precision

if p_bc is not None and confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)) is not None:
    r, p = recall_precision(confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)))
    if r is not None:
        print(f"recall {r:.3f} | precision {p:.3f}")

### C4. Choosing the threshold

**What this cell must do:** for thresholds $t = 0.5, 0.3, 0.1$, print the number of **missed cancers** (FN), **false alarms** (FP), recall and precision.

**Your task:** complete the line that builds the predictions `y_hat` for threshold `t`.

**Hint:** `(p_bc >= t).astype(int)`

**Expected result:**

| t | missed | false alarms | recall | precision |
|---|---|---|---|---|
| 0.5 | 4 | 1 | 0.925 | 0.980 |
| 0.3 | 2 | 1 | 0.962 | 0.981 |
| 0.1 | 1 | 6 | 0.981 | 0.897 |

In [ ]:
if p_bc is not None and confusion_2x2(yb_test, (p_bc >= 0.5).astype(int)) is not None:
    for t in [0.5, 0.3, 0.1]:
        y_hat = None   # TODO
        if y_hat is None:
            break
        cm = confusion_2x2(yb_test, y_hat)
        r, p = recall_precision(cm)
        print(f"t = {t}: missed {cm[1, 0]} | false alarms {cm[0, 1]} | recall {r:.3f} | precision {p:.3f}")

**Question C4:** a hospital says that missing a cancer is about 20 times worse than a false alarm. Using the rule from the lecture, $t^* = C_{FP}/(C_{FP} + C_{FN})$, which threshold should it use? Which of the three thresholds above is closest, and why should the final threshold be chosen on **validation** data rather than on this test set?

**Your answer (C4):**

*Write your answer here — double-click this cell to edit it.*

### C5. Odds ratios: which measurements matter?

**What you need to know:** with standardised features, $e^{w_j}$ is the factor by which the **odds of malignancy** are multiplied when feature $j$ increases by one standard deviation (other features fixed).

**What this cell must do:** take the 30 coefficients of the fitted model, find the five largest, and print their odds ratios.

**Your task:** complete the two `TODO` lines.

**Hint:**
- the logistic-regression step is the last step of the pipeline: `clf[-1].coef_[0]`.
- `pd.Series(values, index=bc.feature_names).sort_values(ascending=False).head(5)`; then `np.exp(...)`.

**Expected result:** worst texture ×3.92, radius error ×3.55, worst symmetry ×2.82, mean concave points ×2.62, worst area ×2.43.

In [ ]:
if clf is not None:
    coefs = None      # TODO: pd.Series of the 30 coefficients, indexed by feature name
    if coefs is not None:
        top5 = coefs.sort_values(ascending=False).head(5)
        odds_ratios = None   # TODO: e^w for the top five
        print(odds_ratios.round(2))

**Question C5:** "compactness error" has a **negative** coefficient. A student concludes that compact nuclei protect against cancer. Explain why this conclusion is not justified.

**Your answer (C5):**

*Write your answer here — double-click this cell to edit it.*

---
# Part D — Softmax regression for three species

Now all three species: **Adelie (0), Chinstrap (1), Gentoo (2)**, still with bill length and depth. You will implement softmax regression with gradient descent — exactly like Part B, with a matrix of weights instead of a vector.

### D0. Data and one-hot labels

**What you need to know:** for gradient descent we encode labels as **one-hot** rows: class 0 → `[1, 0, 0]`, class 1 → `[0, 1, 0]`, class 2 → `[0, 0, 1]`. Row $i$ of the identity matrix `np.eye(3)` is exactly the one-hot vector of class $i$.

**What this cell must do:** build the design matrices `A3_train`, `A3_test` (standardised + bias, already written) and the one-hot matrix `Y_train`.

**Your task:** complete `Y_train`.

**Hint:** `np.eye(3)[y3_train]` — indexing the identity matrix with an array of class numbers picks one row per example.

**Expected result:** `A3_train (256, 3) | Y_train (256, 3) | class counts [113  51  92]`

In [ ]:
classes = np.array(["Adelie", "Chinstrap", "Gentoo"])
X3 = penguins[["bill_length_mm", "bill_depth_mm"]].to_numpy()
y3 = np.searchsorted(classes, penguins["species"].to_numpy())     # 0, 1, 2
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.25, random_state=0, stratify=y3)
mu3, sd3 = X3_train.mean(axis=0), X3_train.std(axis=0)
A3_train = np.c_[np.ones(len(X3_train)), (X3_train - mu3) / sd3]
A3_test = np.c_[np.ones(len(X3_test)), (X3_test - mu3) / sd3]

Y_train = None   # TODO: one-hot matrix of y3_train
if Y_train is not None:
    print("A3_train", A3_train.shape, "| Y_train", Y_train.shape, "| class counts", Y_train.sum(axis=0).astype(int))

### D1. A stable softmax

**What you need to know:** $\text{softmax}(\mathbf{z})_k = e^{z_k}/\sum_j e^{z_j}$. Subtracting the largest score in each row first gives the **same** result and prevents overflow.

**What this cell must do:** define `softmax(Z)` for a matrix `Z` of shape (n, K) — one row of scores per example — and test it on two rows, the second with huge scores.

**Your task:** complete the three lines.

**Hint:** `Z.max(axis=1, keepdims=True)` keeps the shape (n, 1) so that broadcasting subtracts each row's maximum from that row. Same idea for the sum.

**Expected result:**
```
[[0.659  0.2424 0.0986]
 [0.09   0.2447 0.6652]]
```
with no warning — the second row is the same as softmax of `[0, 1, 2]`.

In [ ]:
def softmax(Z):
    Z = Z          # TODO: subtract the row maximum
    E = None       # TODO: exponentiate
    return None    # TODO: divide each row by its sum

Z_test = np.array([[2.0, 1.0, 0.1], [1000.0, 1001.0, 1002.0]])
if softmax(Z_test) is not None:
    print(softmax(Z_test))

In [ ]:
# CHECK D1 — do not edit
assert softmax(Z_test) is not None, "Complete softmax first."
S = softmax(Z_test)
assert S.shape == (2, 3) and np.allclose(S.sum(axis=1), 1), "Each row must sum to 1."
assert np.allclose(S, [[0.65900114, 0.24243297, 0.09856589], [0.09003057, 0.24472847, 0.66524096]])
print("✓ D1")

### D2. The cross-entropy loss

**What you need to know:** $J = -\dfrac{1}{n}\sum_i\sum_k Y_{ik}\log P_{ik}$. Because $Y$ is one-hot, for each example only the log-probability of the **true** class survives.

**What this cell must do:** define `cross_entropy(Y, P)` and evaluate it for a model that predicts $1/3$ for every class.

**Your task:** complete the formula.

**Hint:** `-np.mean(np.sum(Y * np.log(P), axis=1))`.

**Expected result:** `1.0986` $= \log 3$ — the starting loss of gradient descent from $W = 0$.

In [ ]:
def cross_entropy(Y, P, eps=1e-12):
    P = np.clip(P, eps, 1)
    return None   # TODO

P_uniform = np.full((5, 3), 1 / 3)
Y_small = np.eye(3)[[0, 1, 2, 0, 1]]
if cross_entropy(Y_small, P_uniform) is not None:
    print(round(cross_entropy(Y_small, P_uniform), 4))

In [ ]:
# CHECK D2 — do not edit
assert cross_entropy(Y_small, P_uniform) is not None, "Complete cross_entropy first."
assert np.isclose(cross_entropy(Y_small, P_uniform), np.log(3))
P_rand = softmax(np.random.default_rng(1).normal(size=(5, 3)))
assert np.isclose(cross_entropy(Y_small, P_rand), log_loss([0, 1, 2, 0, 1], P_rand))
print("✓ D2 — matches scikit-learn's log_loss")

### D3. Softmax regression by gradient descent

**What you need to know:** with weights $W$ of shape (3 inputs × 3 classes), probabilities $P = \text{softmax}(AW)$ and the gradient
$$\nabla_W J = \frac{1}{n}A^\top(P - Y) \;+\; \lambda W_{\text{no bias row}}.$$
Same pattern as Part B: input-transpose times (prediction − truth).

**What this cell must do:** define `fit_softmax(A, Y, eta, steps, lam)` and train with $\eta = 0.5$, 3000 steps, $\lambda = 1/n$ (equivalent to scikit-learn's `C=1`).

**Your task:** complete the three `TODO` lines in the loop.

**Hint:**
- `P = softmax(A @ W)`
- `G = A.T @ (P - Y) / n`; the penalty line (already written) adds $\lambda W$ to all rows except the bias row 0.
- `W = W - eta * G`

**Expected result:** final training loss `0.1207`.

In [ ]:
def fit_softmax(A, Y, eta=0.5, steps=3000, lam=0.0):
    n, K = Y.shape
    W = np.zeros((A.shape[1], K))
    losses = []
    for step in range(steps):
        P = None                  # TODO: probabilities
        G = None                  # TODO: gradient of the cross-entropy
        if G is None:
            return None, None
        G[1:] += lam * W[1:]      # L2 penalty, not on the bias row
        W = W                     # TODO: update
        losses.append(cross_entropy(Y, P))
    return W, losses

W_gd, losses_sm = (None, None)
if Y_train is not None and softmax(Z_test) is not None:
    W_gd, losses_sm = fit_softmax(A3_train, Y_train, eta=0.5, steps=3000, lam=1 / len(Y_train))
    if W_gd is not None:
        print("final training loss:", round(losses_sm[-1], 4))

**What this cell does:** compares your predicted probabilities on the test set with scikit-learn's multinomial `LogisticRegression(C=1.0)`. (We compare probabilities, not weights: adding the same number to every class's bias gives identical probabilities, so the biases are not unique.)

In [ ]:
# CHECK D3 — do not edit
assert W_gd is not None, "Complete fit_softmax first."
sk3 = LogisticRegression(C=1.0, max_iter=5000).fit((X3_train - mu3) / sd3, y3_train)
diff = np.abs(softmax(A3_test @ W_gd) - sk3.predict_proba((X3_test - mu3) / sd3)).max()
print("largest difference in test probabilities:", round(diff, 4))
assert diff < 0.01
print("✓ D3 — your softmax regression matches scikit-learn")

### D4. Accuracy, confusion matrix and a new penguin

**What this cell must do:** predict the test classes (the column with the largest probability), print the accuracy and the 3 × 3 confusion matrix, and the three probabilities for a new penguin with a 45 mm long, 17 mm deep bill.

**Your task:** complete `pred3` and `x_new`.

**Hint:**
- `softmax(A3_test @ W_gd).argmax(axis=1)`
- the new penguin must be standardised **with the training mean and standard deviation** and get a leading 1: `np.r_[1, (np.array([45, 17]) - mu3) / sd3]`.

**Expected result:** accuracy `0.953`; confusion matrix `[[37 1 0] [1 15 1] [0 1 30]]`; new penguin ≈ `[0.162 0.549 0.289]` → Chinstrap, but only about 55 % sure.

In [ ]:
if W_gd is not None:
    pred3 = None    # TODO
    x_new = None    # TODO
    if pred3 is not None:
        print("test accuracy:", round(np.mean(pred3 == y3_test), 3))
        print(confusion_matrix(y3_test, pred3))
    if x_new is not None:
        print("new penguin:", softmax(x_new[None, :] @ W_gd).round(3), "->", classes[softmax(x_new[None, :] @ W_gd).argmax()])

### D5. Decision regions

**What this cell does:** colours every point of a grid by the predicted species and overlays the test penguins.

**Your task:** just run it.

**Expected result:** three regions separated by **straight** boundaries that meet near (45, 17); almost all test penguins in their own region.

In [ ]:
if W_gd is not None:
    gl, gd = np.meshgrid(np.linspace(31, 61, 300), np.linspace(12.5, 22, 300))
    G3 = np.c_[np.ones(gl.size), (np.c_[gl.ravel(), gd.ravel()] - mu3) / sd3]
    region = softmax(G3 @ W_gd).argmax(axis=1).reshape(gl.shape)
    colours = ["#2a78d6", "#eb6834", "#1baf7a"]
    plt.figure(figsize=(7, 5))
    plt.contourf(gl, gd, region, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#cfe0f5", "#f9d2c1", "#c3eadb"])
    for k in range(3):
        plt.scatter(*X3_test[y3_test == k].T, color=colours[k], edgecolor="white", label=classes[k])
    plt.xlabel("bill length (mm)"); plt.ylabel("bill depth (mm)"); plt.legend(); plt.title("Softmax regression: decision regions")
    plt.show()

**Question D:** all four test errors involve Chinstrap. Using the plot, explain why. What is the recall for Chinstrap, and why is it lower than for the other species?

**Your answer (D):**

*Write your answer here — double-click this cell to edit it.*

---
# Part E — Naive Bayes: Gaussian and a spam filter from scratch

Naive Bayes is **generative**: it models how each class generates its features, then uses Bayes' theorem. Training is just **counting** and **averaging** — no gradient descent.

### E1. Gaussian Naive Bayes on the penguins

**What you need to know:** for each class $k$ estimate the prior $P(y=k)$ (fraction of training examples), and for each feature $j$ a mean $\mu_{jk}$ and a variance $\sigma^2_{jk}$. Predict with
$$\hat y = \arg\max_k \Big[\log P(y=k) + \sum_j \log \mathcal{N}(x_j;\ \mu_{jk}, \sigma^2_{jk})\Big],\qquad \log\mathcal{N}(x;\mu,\sigma^2) = -\tfrac12\log(2\pi\sigma^2) - \tfrac{(x-\mu)^2}{2\sigma^2}.$$
We use the **raw** measurements in mm (no scaling needed — each feature gets its own mean and variance).

**What this cell must do:** estimate `priors` (3,), `means` (3, 2) and `variances` (3, 2) from `X3_train`, `y3_train`, then define `gnb_predict(X)`.

**Your task:** complete the three estimates and the log-density line.

**Hint:** `X3_train[y3_train == k]` selects the training rows of class `k`; use `.mean(axis=0)` and `.var(axis=0)`.

**Expected result:** priors `[0.441 0.199 0.359]`; means ≈ `[[39.01 18.47] [48.99 18.44] [47.44 14.91]]`; test accuracy `0.93`.

In [ ]:
priors = None      # TODO: np.array of the 3 class fractions
means = None       # TODO: (3, 2) array of class means
variances = None   # TODO: (3, 2) array of class variances

def gnb_predict(X):
    scores = []
    for k in range(3):
        log_density = None   # TODO: (n, 2) array of log N(x_j; mean, variance) for class k
        scores.append(np.log(priors[k]) + log_density.sum(axis=1))
    return np.argmax(np.stack(scores, axis=1), axis=1)

if variances is not None:
    print("priors", priors.round(3)); print("means\n", means.round(2))
    print("test accuracy:", round(np.mean(gnb_predict(X3_test) == y3_test), 3))

In [ ]:
# CHECK E1 — do not edit
assert variances is not None, "Complete E1 first."
gnb = GaussianNB().fit(X3_train, y3_train)
assert np.allclose(means, gnb.theta_) and np.allclose(priors, gnb.class_prior_)
assert np.array_equal(gnb_predict(X3_test), gnb.predict(X3_test)), "Your predictions differ from scikit-learn's GaussianNB."
print("✓ E1 — identical predictions to scikit-learn")

**Question E1:** Gaussian Naive Bayes reaches 93.0 % here, softmax regression 95.3 %. The "naive" assumption says bill length and depth are independent **within each species**. Is that plausible for penguins? How does the assumption show up in the lecture's plot of the fitted Gaussians?

**Your answer (E1):**

*Write your answer here — double-click this cell to edit it.*

### E2. The SMS spam data and the bag of words

**What this cell does:**
1. Downloads the **SMS Spam Collection** (5,572 real messages, 747 spam).
2. Splits 75 % / 25 % with `random_state=42`, keeping the spam fraction.
3. Turns each message into **word counts** with `CountVectorizer`, fitted on the **training** messages only. `C_train` is a sparse matrix: one row per message, one column per word of the vocabulary.

**Your task:** just run it.

**Expected result:** `C_train (4179, 7387) | C_test (1393, 7387) | spam fraction 0.134`

In [ ]:
sms_url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
sms = pd.read_csv(sms_url, sep="\t", header=None, names=["label", "message"])
y_sms = (sms["label"] == "spam").astype(int).to_numpy()
m_train, m_test, ys_train, ys_test = train_test_split(sms["message"], y_sms, test_size=0.25, random_state=42, stratify=y_sms)

vectorizer = CountVectorizer()
C_train = vectorizer.fit_transform(m_train)
C_test = vectorizer.transform(m_test)
vocab = vectorizer.get_feature_names_out()
print("C_train", C_train.shape, "| C_test", C_test.shape, "| spam fraction", round(ys_train.mean(), 3))

### E3. Multinomial Naive Bayes: training is counting

**What you need to know:**
- log prior: $\log P(c)$ = log of the fraction of training messages in class $c$.
- word probabilities with **Laplace smoothing**: $\hat P(w\mid c) = \dfrac{\text{count}(w,c)+\alpha}{\sum_v\text{count}(v,c)+\alpha|V|}$, where count$(w, c)$ is how often word $w$ appears in all training messages of class $c$.

**What this cell must do:** define `mnb_fit(C, y, alpha)` returning `log_prior` (2,) and `log_word_prob` (2, |V|), and fit it with $\alpha = 1$.

**Your task:** complete the two `TODO` lines (the counts are already computed).

**Hint:**
- `counts` has shape (2, |V|); `counts.sum(axis=1, keepdims=True)` is the total number of words in each class, shape (2, 1).
- `C.shape[1]` is $|V|$.

**Expected result:** `log_prior = [-0.1439 -2.0099]` (that is $P(\text{ham}) = 0.866$, $P(\text{spam}) = 0.134$).

In [ ]:
def mnb_fit(C, y, alpha=1.0):
    log_prior = None   # TODO: np.log of [fraction of ham, fraction of spam]
    counts = np.vstack([np.asarray(C[y == k].sum(axis=0)).ravel() for k in [0, 1]])   # (2, |V|)
    with np.errstate(divide="ignore"):           # alpha = 0 is allowed for the experiment in E5
        log_word_prob = None   # TODO: log of the smoothed word probabilities
    return log_prior, log_word_prob

log_prior, log_word_prob = mnb_fit(C_train, ys_train, alpha=1.0)
if log_prior is not None:
    print("log_prior =", log_prior)

In [ ]:
# CHECK E3 — do not edit
assert log_prior is not None and log_word_prob is not None, "Complete mnb_fit first."
nb_sk = MultinomialNB(alpha=1.0).fit(C_train, ys_train)
assert np.allclose(log_prior, nb_sk.class_log_prior_)
assert log_word_prob.shape == (2, C_train.shape[1]) and np.allclose(log_word_prob, nb_sk.feature_log_prob_)
print("✓ E3 — identical to scikit-learn's MultinomialNB")

### E4. Predicting in log space

**What you need to know:** for a message with word-count vector $\mathbf{x}$, the log score of class $c$ is $\log P(c) + \sum_w x_w \log\hat P(w\mid c)$ — a matrix product! For all test messages at once: `C_test @ log_word_prob.T + log_prior`, shape (n, 2).

**What this cell must do:** define `mnb_log_scores(C, log_prior, log_word_prob)` and predict the class with the larger score; print accuracy and confusion matrix.

**Your task:** complete the score line and the prediction.

**Hint:** `np.asarray(...)` turns the sparse-matrix result into an ordinary array; then `.argmax(axis=1)`.

**Expected result:** accuracy `0.9864`; confusion matrix `[[1204 2] [17 170]]` — 2 genuine messages blocked, 17 spam messages missed.

In [ ]:
def mnb_log_scores(C, log_prior, log_word_prob):
    return None   # TODO

if log_prior is not None and mnb_log_scores(C_test, log_prior, log_word_prob) is not None:
    pred_nb = None   # TODO: class with the larger score
    if pred_nb is not None:
        print("test accuracy:", round(np.mean(pred_nb == ys_test), 4))
        print(confusion_matrix(ys_test, pred_nb))

In [ ]:
# CHECK E4 — do not edit
assert log_prior is not None and mnb_log_scores(C_test, log_prior, log_word_prob) is not None, "Complete E3 and E4 first."
assert np.array_equal(mnb_log_scores(C_test, log_prior, log_word_prob).argmax(axis=1), nb_sk.predict(C_test))
print("✓ E4 — your spam filter makes exactly the same predictions as scikit-learn")

### E5. Why smoothing matters

**What this cell does:** the word **"claim"** appears 85 times in training spam and **never** in ham. The cell scores the friendly message *"Did you claim it? Call me when you get home"* without smoothing ($\alpha = 0$) and with $\alpha = 1$.

**What you need to know:** with two log scores $s_{\text{ham}}, s_{\text{spam}}$, the probability of spam is $P(\text{spam}) = \dfrac{1}{1 + e^{\,s_{\text{ham}} - s_{\text{spam}}}} = \sigma(s_{\text{spam}} - s_{\text{ham}})$.

**Your task:** complete the line `p_spam = None`.

**Hint:** use your `sigmoid`.

**Expected result:** with $\alpha = 0$: the ham score is `-inf` and P(spam) = `1.0`; with $\alpha = 1$: P(spam) ≈ `0.0001`.

In [ ]:
msg = ["Did you claim it? Call me when you get home"]
if log_prior is not None and mnb_log_scores(C_test, log_prior, log_word_prob) is not None:
    x_msg = vectorizer.transform(msg)
    for alpha in [0.0, 1.0]:
        lp, lwp = mnb_fit(C_train, ys_train, alpha=alpha)
        x_dense = x_msg.toarray()[0]
        used = x_dense > 0                          # only words that occur in the message
        s = lp + (lwp[:, used] * x_dense[used]).sum(axis=1)
        p_spam = None    # TODO: sigmoid of (spam score - ham score)
        if p_spam is not None:
            print(f"alpha = {alpha}: scores (ham, spam) = {s.round(2)} | P(spam) = {p_spam:.4f}")

**What this cell does:** tries five values of $\alpha$ on the test set and prints accuracy, blocked genuine messages (FP) and missed spam (FN). (Only for exploration — see the question below.)

**Your task:** just run it.

**Expected result:** α = 1e-10: 0.9821 · 0.01: 0.9878 · 0.1: 0.9885 · 1: 0.9864 · 3: 0.9785.

In [ ]:
if log_prior is not None and mnb_log_scores(C_test, log_prior, log_word_prob) is not None:
    for alpha in [1e-10, 0.01, 0.1, 1.0, 3.0]:
        lp, lwp = mnb_fit(C_train, ys_train, alpha=alpha)
        pr = mnb_log_scores(C_test, lp, lwp).argmax(axis=1)
        print(f"alpha = {alpha:<6} accuracy {np.mean(pr == ys_test):.4f} | FP {np.sum((pr == 1) & (ys_test == 0))} | FN {np.sum((pr == 0) & (ys_test == 1))}")

**Question E5:** (a) Explain in your own words why the unsmoothed model is 100 % sure that the friendly message is spam. (b) The table suggests α = 0.1 is slightly better than α = 1 on the test set. Should we now report "98.85 % accuracy with α = 0.1"? Why not, and what should we do instead?

**Your answer (E5):**

*Write your answer here — double-click this cell to edit it.*

### E6. What did the filter learn?

**What you need to know:** the **log-ratio** $\log\hat P(w\mid\text{spam}) - \log\hat P(w\mid\text{ham})$ says how much more typical a word is of spam. Large positive = strong spam evidence.

**What this cell must do:** compute the log-ratio for every word and print the 10 words with the largest values.

**Your task:** complete the two `TODO` lines.

**Hint:** `log_word_prob[1] - log_word_prob[0]`; `np.argsort(ratio)[::-1][:10]` gives the indices of the 10 largest; index `vocab` with them.

**Expected result:** `['claim' 'prize' '150p' 'tone' 'guaranteed' '500' 'cs' '18' '1000' 'awarded']` (the order of tied words may differ).

In [ ]:
if log_prior is not None:
    ratio = None      # TODO
    top10 = None      # TODO: the 10 words with the largest ratio
    if top10 is not None:
        print(top10)

### E7. Naive Bayes vs logistic regression vs the baseline

**What this cell does:** trains scikit-learn's logistic regression on the same word counts and prints its accuracy, together with the "always ham" baseline.

**Your task:** just run it, then answer the question.

**Expected result:** logistic regression `0.9785`, baseline `0.8658`; your Naive Bayes was `0.9864`.

In [ ]:
lr_sms = LogisticRegression(max_iter=5000).fit(C_train, ys_train)
print("logistic regression:", round(lr_sms.score(C_test, ys_test), 4))
print("always 'ham' baseline:", round(1 - ys_test.mean(), 4))
print(confusion_matrix(ys_test, lr_sms.predict(C_test)))

**Question E7:** logistic regression blocks **no** genuine messages but misses 30 spams; Naive Bayes blocks 2 genuine messages and misses 17. For a phone company's spam filter, which mistakes matter more? Which model would you deploy, and what would you change?

**Your answer (E7):**

*Write your answer here — double-click this cell to edit it.*

---
# Part F — Reflection

Answer each question in 3–6 sentences, in your own words.

**F1.** In Parts B and D the gradient had the same form: $\frac{1}{n}A^\top(\text{prediction} - \text{truth})$. Where did this form come from, and which loss gave the same form in Week 2?

**Your answer (F1):**

*Write your answer here — double-click this cell to edit it.*

**F2.** Logistic regression needed many gradient-descent steps; Naive Bayes was trained by counting in one pass. Explain this difference using the words *discriminative* and *generative*.

**Your answer (F2):**

*Write your answer here — double-click this cell to edit it.*

**F3.** Give one example from today's lab where the default threshold of 0.5 is a poor choice, and explain how you would choose a better one.

**Your answer (F3):**

*Write your answer here — double-click this cell to edit it.*

**F4.** Which part of today's lab was the most difficult for you, and what helped you (or would have helped you) to understand it?

**Your answer (F4):**

*Write your answer here — double-click this cell to edit it.*

---
## Submission checklist

- [ ] **Restart and run all** — the notebook runs top to bottom with no errors.
- [ ] All **13 CHECK cells** print ✓ (A1, A2, A3, A4, B1, C1, C2, D1, D2, D3, E1, E3, E4).
- [ ] All plots are visible (B3, D5).
- [ ] All written answers are completed (A3, A4, B4, B5, C4, C5, D, E1, E5, E7, F1–F4).
- [ ] File renamed to `Lab03_<Surname>_<Name>.ipynb` and uploaded.

**Marking (100 points):** Part A 15 · Part B 20 · Part C 20 · Part D 20 · Part E 20 · Part F and presentation 5.